# Lab 15-05 · Stability and the Routh–Hurwitz Criterion

Four parts, each built on `routh.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from routh import routh, first_column, rhp_count, jw_roots

## Part 1 · The criterion against computed roots

*Predict:* will the Routh count ever disagree with the roots?

In [2]:
rng = np.random.default_rng(5)
disagree = 0
for trial in range(500):
    n = rng.integers(3, 8)
    roots = rng.normal(size=n) * 2 + 1j * 0
    # build real polynomials from random real roots and conjugate pairs
    k = n // 2
    pairs = rng.normal(-0.2, 1.5, size=k) + 1j * np.abs(rng.normal(0, 2, size=k))
    reals = rng.normal(-0.2, 1.5, size=n - 2 * k)
    allr = np.concatenate([pairs, pairs.conj(), reals])
    c = np.real(np.poly(allr))
    if np.min(np.abs(allr.real)) < 1e-3:
        continue                      # skip polynomials with roots too close to the axis
    disagree += rhp_count(c) != int(np.sum(allr.real > 0))
print("disagreements in 500 random polynomials:", disagree)
for c in ([1, 1, 2, 2, 3], [1, 2, 6, 8, 8], [1, 1, 6, 6, 8, 8]):
    rows, notes, aux = routh(c)
    print(c, "first column", np.round(first_column(c), 3), "RHP", rhp_count(c), notes, "axis roots", None if aux is None else np.round(np.roots(aux), 4))

disagreements in 500 random polynomials: 0
[1, 1, 2, 2, 3] first column [ 1.e+00  1.e+00  0.e+00 -3.e+09  3.e+00] RHP 2 ['row s^2 had a zero first element: replaced by eps'] axis roots None
[1, 2, 6, 8, 8] first column [1. 2. 2. 4. 8.] RHP 0 ['row s^1 was all zero: replaced by d/ds of auxiliary [2.0, 0.0, 8.0]'] axis roots [-0.+2.j  0.-2.j]
[1, 1, 6, 6, 8, 8] first column [1.    1.    4.    3.    1.333 8.   ] RHP 0 ['row s^3 was all zero: replaced by d/ds of auxiliary [1.0, 0.0, 6.0, 0.0, 8.0]'] axis roots [-0.+2.j     -0.-2.j      0.+1.4142j  0.-1.4142j]


## Part 2 · A gain sweep

In [3]:
Ks = np.linspace(1, 80, 400)
R = np.array([np.roots([1, 6, 8, K]) for K in Ks])
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(R.real, R.imag, ".", ms=2)
ax.axvline(0, color="gray", lw=0.6)
r48 = np.roots([1, 6, 8, 48])
ax.plot(r48.real, r48.imag, "rx", ms=8, label="K = 48")
ax.set_xlabel("Re s"); ax.set_ylabel("Im s"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
print("roots at K = 48:", np.round(r48, 4))

roots at K = 48: [-6.+0.j      0.+2.8284j  0.-2.8284j]


## Part 3 · The ring oscillator

*Predict:* does the ring oscillate at the linear model's frequency or near $\sqrt3 f_0$?

In [4]:
Rr, C, gm, Vs = 1e3, 1e-9, 2.2e-3, 0.1
A, w0 = gm * Rr, 1 / (Rr * C)
lin = np.roots((np.poly1d([1 / w0, 1]) ** 3 + A**3).coeffs)
p = lin[np.argmax(lin.imag)]
print(f"linear model: pole {p.real:.3e} + j{p.imag:.4e}  ->  {p.imag / 2 / math.pi / 1e3:.1f} kHz, growing")
def rhs(t, v):
    f = lambda x: -gm * Vs * np.tanh(x / Vs)
    return [(-v[0] / Rr + f(v[2])) / C, (-v[1] / Rr + f(v[0])) / C, (-v[2] / Rr + f(v[1])) / C]
sol = solve_ivp(rhs, (0, 200e-6), [1e-3, 0, 0], dense_output=True, rtol=1e-10, atol=1e-13)
t = np.linspace(0, 200e-6, 400001)
v = sol.sol(t)[0]
up = t[1:][(v[:-1] < 0) & (v[1:] >= 0)]
print(f"steady frequency (Python): {10 / (up[49] - up[39]) / 1e3:.1f} kHz;  sqrt(3) f0 = {math.sqrt(3) * w0 / 2 / math.pi / 1e3:.1f} kHz")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t * 1e6, v * 1e3)
ax.set_xlabel("t (µs)"); ax.set_ylabel("v1 (mV)"); ax.grid(True, alpha=0.3)
plt.show()
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "ring3.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith("fosc")])
else:
    print("ngspice not installed: skip (CI runs ring3.cir through tools/run_spice.py)")

linear model: pole 1.000e+05 + j1.9053e+06  ->  303.2 kHz, growing


steady frequency (Python): 276.2 kHz;  sqrt(3) f0 = 275.7 kHz
ngspice: ['fosc = 2.761958e+05']


## Part 4 · Relative stability

In [5]:
def shifted(K, sigma=0.5):
    return (np.poly1d([1, 6, 8, K])(np.poly1d([1, -sigma]))).coeffs
ok = [K for K in np.linspace(0.5, 30, 2951) if rhp_count(shifted(K)) == 0]
print(f"all poles left of -0.5 for K in about [{ok[0]:.2f}, {ok[-1]:.2f}]")

all poles left of -0.5 for K in about [2.63, 15.00]


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.